### 슬라이드 분석 모듈

#### 환경 설정

In [19]:
# library import
import os
from dotenv import load_dotenv
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 환경 변수
load_dotenv()
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

#### Structured Output 정의

In [20]:
# Slide 모델 정의
class Slide(BaseModel):
    slide_number: int = Field(description="슬라이드 번호. 1부터 시작하는 정수")
    script: str = Field(description="해당 슬라이드에 포함된 발표 대본")
    keywords: list[str] = Field(description="해당 슬라이드 대본에 포함된 키워드 목록")

# 전체 결과 모델 정의
class SeperatedSlides(BaseModel):
    status: Literal["success", "fail"] = Field(description="슬라이드 구분 성공 여부")
    slides: list[Slide] = Field(description="슬라이드별 발표 대본 목록")

#### System Prompt 작성

In [21]:
SYSTEM_PROMPT = """
너는 발표 대본을 슬라이드별로 분리하는 전문가야.
입력은 하나의 발표 대본 전체 텍스트야.

# 슬라이드 구분 판단 기준
- "Slide", "슬라이드" 또는 숫자같은 명시적 표기만 슬라이드 구분으로 인정해.
- "첫째", "둘째", "다음으로" 같은 서수/전환 표현은 슬라이드 구분이 아니야.
- 대본 전체에서 이런 명시적 구분이 하나도 없거나, 일부 구간에만 있고 나머지 구간은 구분할 수 없으면 반드시 status="fail"로 처리해. 임의로 슬라이드 번호를 만들어내거나 추측해서 나누지 마.

# status="success"인 경우
- 대본 전체가 명시적 구분자를 기준으로 빠짐없이 나뉠 수 있을 때만 success로 판단해.
- slides 배열에 모든 슬라이드를 slide_number, script와 함께 순서대로 채워.
- slide_number는 원문에 표기된 번호를 그대로 사용해. (원문에 1, 3, 5만 있으면 그대로 1, 3, 5로 채워.)
- script에는 구분 기호(예: "1.", "Slide 2:") 자체는 제거하고, 그 슬라이드에 해당하는 본문 텍스트만 담아.
- keywords에는 해당 슬라이드 대본에서 추출한 중요한 단어 목록을 3~7개 정도 담아. (키워드 추출은 자유롭게 해도 돼.)
- keywords의 순서는 대본에 등장하는 순서대로 담아. (중복 단어는 제거해)
- keywords에는 단순 예시가 나열된 경우, 의미 없는 단어, 조사/접속사 등은 포함하지 마.

# status="fail"인 경우
- slides 필드는 빈 리스트로 반환해.
"""

#### LLM 모델 불러오기

In [22]:
os.getenv('OPENAI_MODEL'), os.getenv('OPENAI_BASE_URL')

('openai/gpt-5.6-luna',
 'https://mlapi.run/286e9158-d32e-436d-a23d-36b43fc8e68a/v1')

In [23]:
# LLM 객체 생성
model = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
)

# Structured Output 연결
structured_output = model.with_structured_output(SeperatedSlides)

# 실제 모델 호출이 되는지 간단한 테스트
response = structured_output.invoke("Please separate the following script into slides:\n\n안녕하세요. 오늘은 멸종위기청년에 대해 발표하겠습니다.") 
response

SeperatedSlides(status='success', slides=[Slide(slide_number=1, script='안녕하세요. 오늘은 멸종위기청년에 대해 발표하겠습니다.', keywords=['멸종위기청년', '발표'])])

#### 테스트용 대본 파일 import

In [24]:
scripts = []

# 테스트용 .txt 파일 읽기
path = os.path.join(os.getcwd(), "대본")
for filename in os.listdir(path):
    if filename.endswith(".txt"):
        with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
            script = file.read()
            scripts.append(script)
            
scripts

['[슬라이드 1]\n안녕하세요. 저희는 발표 연습을 도와주는 AI 서비스, 피치코치를 소개하겠습니다.\n\n발표를 준비할 때 우리는 보통 거울을 보거나 혼자 녹화하면서 연습합니다. 하지만 내가 실제로 너무 빨리 말하고 있는지, 시선을 제대로 처리하고 있는지는 혼자 확인하기 어렵습니다.\n\n[슬라이드 2]\n저희가 주목한 문제는 크게 세 가지입니다.\n\n첫째, 발표 속도와 목소리 크기를 객관적으로 확인하기 어렵습니다.\n둘째, 발표 중 시선이나 불필요한 말버릇을 스스로 발견하기 어렵습니다.\n셋째, 연습을 반복해도 무엇을 개선해야 하는지 명확하지 않습니다.\n\n[슬라이드 3]\n피치코치는 발표자가 PPT와 대본을 등록하고 직접 발표를 연습할 수 있도록 합니다.\n\n발표하는 동안 음성과 카메라를 분석하고, 문제가 일정 시간 이상 지속되면 AI가 실시간으로 피드백을 제공합니다.\n\n[슬라이드 4]\n예를 들어 발표자가 지나치게 빠르게 말하면 "조금만 천천히 말해보세요"라는 피드백을 제공합니다.\n\n시선이 화면에 지나치게 오래 머무른다면 청중을 바라보도록 안내할 수도 있습니다.\n\n다만 모든 순간에 피드백을 주지는 않습니다. 순간적인 실수보다 반복적으로 나타나는 문제를 중심으로 개입합니다.\n\n[슬라이드 5]\n발표가 끝나면 최종 리포트를 제공합니다.\n\n발표 시간, 말하기 속도, pause, 목소리 크기, 시선 처리, filler word 사용량 등의 변화를 확인할 수 있습니다.\n\n또한 이전 발표와 비교하여 어떤 부분이 개선되었는지도 보여줍니다.\n\n[슬라이드 6]\n결국 피치코치가 해결하고자 하는 문제는 단순히 발표를 평가하는 것이 아닙니다.\n\n발표자가 자신의 문제를 발견하고, 다음 연습에서 개선하고, 다시 확인하는 반복적인 연습 과정을 만드는 것입니다.\n\n이상으로 발표를 마치겠습니다.',
 'K-pop의 글로벌 성공\n\nSlide 01: K-pop은 어떻게 세계적인 장르가 되었을까?\n\n안녕하세요. 오늘은 K-pop이 한국을 넘

#### 전처리 (마크다운/개행 제거)

In [25]:
import re

def clean_script(text: str) -> str:
    # 헤더(#, ##, ...) 제거
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    # 굵게/기울임 강조 기호 제거: **text**, __text__, *text*, _text_
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    text = re.sub(r"__(.+?)__", r"\1", text)
    text = re.sub(r"(?<!\w)\*(.+?)\*(?!\w)", r"\1", text)
    text = re.sub(r"(?<!\w)_(.+?)_(?!\w)", r"\1", text)
    # 인라인 코드 백틱(`code`) 제거
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # 마크다운 링크 [텍스트](URL) -> 텍스트 (슬라이드 마커 [슬라이드 1] 등은 뒤에 괄호가 없어 영향 없음)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # 개행 및 연속 공백을 단일 공백으로 정리
    text = re.sub(r"\s+", " ", text)
    return text.strip()

scripts = [clean_script(script) for script in scripts]
scripts

['[슬라이드 1] 안녕하세요. 저희는 발표 연습을 도와주는 AI 서비스, 피치코치를 소개하겠습니다. 발표를 준비할 때 우리는 보통 거울을 보거나 혼자 녹화하면서 연습합니다. 하지만 내가 실제로 너무 빨리 말하고 있는지, 시선을 제대로 처리하고 있는지는 혼자 확인하기 어렵습니다. [슬라이드 2] 저희가 주목한 문제는 크게 세 가지입니다. 첫째, 발표 속도와 목소리 크기를 객관적으로 확인하기 어렵습니다. 둘째, 발표 중 시선이나 불필요한 말버릇을 스스로 발견하기 어렵습니다. 셋째, 연습을 반복해도 무엇을 개선해야 하는지 명확하지 않습니다. [슬라이드 3] 피치코치는 발표자가 PPT와 대본을 등록하고 직접 발표를 연습할 수 있도록 합니다. 발표하는 동안 음성과 카메라를 분석하고, 문제가 일정 시간 이상 지속되면 AI가 실시간으로 피드백을 제공합니다. [슬라이드 4] 예를 들어 발표자가 지나치게 빠르게 말하면 "조금만 천천히 말해보세요"라는 피드백을 제공합니다. 시선이 화면에 지나치게 오래 머무른다면 청중을 바라보도록 안내할 수도 있습니다. 다만 모든 순간에 피드백을 주지는 않습니다. 순간적인 실수보다 반복적으로 나타나는 문제를 중심으로 개입합니다. [슬라이드 5] 발표가 끝나면 최종 리포트를 제공합니다. 발표 시간, 말하기 속도, pause, 목소리 크기, 시선 처리, filler word 사용량 등의 변화를 확인할 수 있습니다. 또한 이전 발표와 비교하여 어떤 부분이 개선되었는지도 보여줍니다. [슬라이드 6] 결국 피치코치가 해결하고자 하는 문제는 단순히 발표를 평가하는 것이 아닙니다. 발표자가 자신의 문제를 발견하고, 다음 연습에서 개선하고, 다시 확인하는 반복적인 연습 과정을 만드는 것입니다. 이상으로 발표를 마치겠습니다.',
 'K-pop의 글로벌 성공 Slide 01: K-pop은 어떻게 세계적인 장르가 되었을까? 안녕하세요. 오늘은 K-pop이 한국을 넘어 세계적인 음악 장르로 성장한 이유를 살펴보겠습니다. 몇 년 전까지만 해도 한국 음악은 주로 아시아

#### 단일 대본 test

In [26]:
result = structured_output.invoke(
    [
        ("system", SYSTEM_PROMPT),
        ("user", scripts[0])
    ]
)
data = result.model_dump()
data

{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '안녕하세요. 저희는 발표 연습을 도와주는 AI 서비스, 피치코치를 소개하겠습니다. 발표를 준비할 때 우리는 보통 거울을 보거나 혼자 녹화하면서 연습합니다. 하지만 내가 실제로 너무 빨리 말하고 있는지, 시선을 제대로 처리하고 있는지는 혼자 확인하기 어렵습니다.',
   'keywords': ['발표 연습', 'AI 서비스', '피치코치', '발표 속도', '시선 처리']},
  {'slide_number': 2,
   'script': '저희가 주목한 문제는 크게 세 가지입니다. 첫째, 발표 속도와 목소리 크기를 객관적으로 확인하기 어렵습니다. 둘째, 발표 중 시선이나 불필요한 말버릇을 스스로 발견하기 어렵습니다. 셋째, 연습을 반복해도 무엇을 개선해야 하는지 명확하지 않습니다.',
   'keywords': ['발표 속도', '목소리 크기', '시선', '말버릇', '개선 방향']},
  {'slide_number': 3,
   'script': '피치코치는 발표자가 PPT와 대본을 등록하고 직접 발표를 연습할 수 있도록 합니다. 발표하는 동안 음성과 카메라를 분석하고, 문제가 일정 시간 이상 지속되면 AI가 실시간으로 피드백을 제공합니다.',
   'keywords': ['PPT', '대본', '발표 연습', '음성 분석', '카메라 분석', '실시간 피드백']},
  {'slide_number': 4,
   'script': '예를 들어 발표자가 지나치게 빠르게 말하면 "조금만 천천히 말해보세요"라는 피드백을 제공합니다. 시선이 화면에 지나치게 오래 머무른다면 청중을 바라보도록 안내할 수도 있습니다. 다만 모든 순간에 피드백을 주지는 않습니다. 순간적인 실수보다 반복적으로 나타나는 문제를 중심으로 개입합니다.',
   'keywords': ['빠른 말하기', '피드백', '시선 안내', '청중', '반복 문제']},
  {'slide_

#### 전체 대본 test

In [27]:
results = []

for script in scripts:
    result = structured_output.invoke(
        [
            ("system", SYSTEM_PROMPT),
            ("user", script)
        ]
    )
    results.append(result.model_dump())

results

[{'status': 'success',
  'slides': [{'slide_number': 1,
    'script': '안녕하세요. 저희는 발표 연습을 도와주는 AI 서비스, 피치코치를 소개하겠습니다. 발표를 준비할 때 우리는 보통 거울을 보거나 혼자 녹화하면서 연습합니다. 하지만 내가 실제로 너무 빨리 말하고 있는지, 시선을 제대로 처리하고 있는지는 혼자 확인하기 어렵습니다.',
    'keywords': ['발표 연습', 'AI 서비스', '피치코치', '발표 속도', '시선 처리']},
   {'slide_number': 2,
    'script': '저희가 주목한 문제는 크게 세 가지입니다. 첫째, 발표 속도와 목소리 크기를 객관적으로 확인하기 어렵습니다. 둘째, 발표 중 시선이나 불필요한 말버릇을 스스로 발견하기 어렵습니다. 셋째, 연습을 반복해도 무엇을 개선해야 하는지 명확하지 않습니다.',
    'keywords': ['발표 속도', '목소리 크기', '시선', '말버릇', '개선점']},
   {'slide_number': 3,
    'script': '피치코치는 발표자가 PPT와 대본을 등록하고 직접 발표를 연습할 수 있도록 합니다. 발표하는 동안 음성과 카메라를 분석하고, 문제가 일정 시간 이상 지속되면 AI가 실시간으로 피드백을 제공합니다.',
    'keywords': ['피치코치', 'PPT', '대본', '음성 분석', '카메라 분석', '실시간 피드백']},
   {'slide_number': 4,
    'script': '예를 들어 발표자가 지나치게 빠르게 말하면 "조금만 천천히 말해보세요"라는 피드백을 제공합니다. 시선이 화면에 지나치게 오래 머무른다면 청중을 바라보도록 안내할 수도 있습니다. 다만 모든 순간에 피드백을 주지는 않습니다. 순간적인 실수보다 반복적으로 나타나는 문제를 중심으로 개입합니다.',
    'keywords': ['빠른 말하기', '피드백', '시선', '청중', '반복적인 문제']},
 